# US Flight Delays Network Analysis
## Step 2: Graph Construction

**Course:** CS-GY 6513-C Big Data — Spring 2026

**Team:** Mirsaid Abbasov [ma9197], Nicholas Pesa [np2354], Ferdi Fadillah [ff2364]

---

In this step we transform the cleaned BTS data into a directed graph:
- **Nodes:** airports (enriched with OpenFlights metadata — name, city, lat/lon)
- **Edges:** directed routes (origin → destination) with delay-based weights

**Inputs:**
- Cleaned BTS Parquet from Step 1
- OpenFlights `airports.dat` for airport metadata

**Outputs:**
- `nodes.parquet` — airports with traffic & delay metrics
- `edges.parquet` — routes with traffic & delay metrics

## 1. Environment Setup

In [ ]:
!pip install pyspark -q

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, DoubleType

spark = SparkSession.builder \
    .appName("FlightDelays_Step2") \
    .config("spark.driver.memory", "4g") \
    .config("spark.sql.shuffle.partitions", "50") \
    .getOrCreate()

print(f"Spark version: {spark.version}")

Spark version: 4.0.2


## 2. Load Cleaned Flight Data from Step 1

In [ ]:
PARQUET_PATH = "/content/drive/MyDrive/Big Data Project/parquet/completed_flights"

flights = spark.read.parquet(PARQUET_PATH)
print(f"Loaded {flights.count():,} completed flights")
flights.printSchema()

Loaded 30,821,441 completed flights
root
 |-- DAY_OF_WEEK: integer (nullable = true)
 |-- FL_DATE: date (nullable = true)
 |-- OP_UNIQUE_CARRIER: string (nullable = true)
 |-- ORIGIN_AIRPORT_ID: integer (nullable = true)
 |-- ORIGIN: string (nullable = true)
 |-- DEST_AIRPORT_ID: integer (nullable = true)
 |-- DEST: string (nullable = true)
 |-- CRS_DEP_TIME: integer (nullable = true)
 |-- DEP_DELAY: double (nullable = true)
 |-- CRS_ARR_TIME: integer (nullable = true)
 |-- ARR_DELAY: double (nullable = true)
 |-- ARR_DEL15: double (nullable = true)
 |-- CANCELLED: double (nullable = true)
 |-- DIVERTED: double (nullable = true)
 |-- DISTANCE: double (nullable = true)
 |-- CARRIER_DELAY: double (nullable = true)
 |-- WEATHER_DELAY: double (nullable = true)
 |-- NAS_DELAY: double (nullable = true)
 |-- SECURITY_DELAY: double (nullable = true)
 |-- LATE_AIRCRAFT_DELAY: double (nullable = true)
 |-- YEAR: integer (nullable = true)
 |-- MONTH: integer (nullable = true)



## 3. Load OpenFlights Airport Metadata

The `airports.dat` file has no header. Columns (per OpenFlights docs):

`id, name, city, country, iata, icao, latitude, longitude, altitude, timezone, dst, tz_db, type, source`

We only need a few of these.

In [ ]:
AIRPORTS_PATH = "/content/drive/MyDrive/Big Data Project/airports.dat.csv"

airports_schema = StructType([
    StructField("of_id",      IntegerType(), True),
    StructField("name",       StringType(),  True),
    StructField("city",       StringType(),  True),
    StructField("country",    StringType(),  True),
    StructField("iata",       StringType(),  True),
    StructField("icao",       StringType(),  True),
    StructField("latitude",   DoubleType(),  True),
    StructField("longitude",  DoubleType(),  True),
    StructField("altitude",   IntegerType(), True),
    StructField("timezone",   StringType(),  True),
    StructField("dst",        StringType(),  True),
    StructField("tz_db",      StringType(),  True),
    StructField("type",       StringType(),  True),
    StructField("source",     StringType(),  True),
])

airports_raw = spark.read.csv(
    AIRPORTS_PATH,
    schema=airports_schema,
    header=False,
    quote='"',
    escape='"'
)

print(f"Total OpenFlights airports: {airports_raw.count():,}")
airports_raw.show(5, truncate=False)

Total OpenFlights airports: 7,698
+-----+-------------------------------------------+------------+----------------+----+----+------------------+------------------+--------+--------+---+--------------------+-------+-----------+
|of_id|name                                       |city        |country         |iata|icao|latitude          |longitude         |altitude|timezone|dst|tz_db               |type   |source     |
+-----+-------------------------------------------+------------+----------------+----+----+------------------+------------------+--------+--------+---+--------------------+-------+-----------+
|1    |Goroka Airport                             |Goroka      |Papua New Guinea|GKA |AYGA|-6.081689834590001|145.391998291     |5282    |10      |U  |Pacific/Port_Moresby|airport|OurAirports|
|2    |Madang Airport                             |Madang      |Papua New Guinea|MAG |AYMD|-5.20707988739    |145.789001465     |20      |10      |U  |Pacific/Port_Moresby|airport|OurAirports|
|

In [ ]:
# Keep only US airports with a valid IATA code
airports_us = airports_raw \
    .filter(F.col("country") == "United States") \
    .filter(F.col("iata").isNotNull()) \
    .filter(F.col("iata") != "\\N") \
    .filter(F.length(F.col("iata")) == 3) \
    .select("iata", "name", "city", "latitude", "longitude")

print(f"US airports with IATA codes: {airports_us.count():,}")
airports_us.show(5, truncate=False)

US airports with IATA codes: 1,251
+----+--------------------------+-------------+------------------+-------------------+
|iata|name                      |city         |latitude          |longitude          |
+----+--------------------------+-------------+------------------+-------------------+
|BTI |Barter Island LRRS Airport|Barter Island|70.1340026855     |-143.582000732     |
|LUR |Cape Lisburne LRRS Airport|Cape Lisburne|68.87509918       |-166.1100006       |
|PIZ |Point Lay LRRS Airport    |Point Lay    |69.73290253       |-163.0050049       |
|ITO |Hilo International Airport|Hilo         |19.721399307250977|-155.04800415039062|
|ORL |Orlando Executive Airport |Orlando      |28.5455           |-81.332901         |
+----+--------------------------+-------------+------------------+-------------------+
only showing top 5 rows


## 4. Build Edges — Directed Routes with Delay Weights

Each edge = a unique (ORIGIN → DEST) pair. We aggregate per edge:
- `num_flights` — total flights on that route
- `avg_dep_delay` — average departure delay (minutes)
- `avg_arr_delay` — average arrival delay (minutes)
- `pct_delayed_15` — fraction of flights delayed ≥15 min at arrival
- `avg_distance` — route distance (constant, but useful)

In [ ]:
edges = flights.groupBy("ORIGIN", "DEST").agg(
    F.count("*").alias("num_flights"),
    F.avg("DEP_DELAY").alias("avg_dep_delay"),
    F.avg("ARR_DELAY").alias("avg_arr_delay"),
    F.avg("ARR_DEL15").alias("pct_delayed_15"),
    F.avg("DISTANCE").alias("avg_distance")
)

# Rename for graph convention: src, dst
edges = edges.withColumnRenamed("ORIGIN", "src").withColumnRenamed("DEST", "dst")

print(f"Unique routes (edges): {edges.count():,}")
edges.orderBy(F.desc("num_flights")).show(10, truncate=False)

Unique routes (edges): 8,128
+---+---+-----------+------------------+--------------------+-------------------+------------+
|src|dst|num_flights|avg_dep_delay     |avg_arr_delay       |pct_delayed_15     |avg_distance|
+---+---+-----------+------------------+--------------------+-------------------+------------+
|SFO|LAX|54453      |8.016619837290875 |-0.05336712394174793|0.16316823682809029|337.0       |
|LAX|SFO|54442      |8.854487344329746 |2.3762352595422653  |0.18985342199037508|337.0       |
|HNL|OGG|47734      |3.3383961117861483|2.6832446474211253  |0.13265177860644403|100.0       |
|LAX|LAS|47690      |9.671146990983434 |5.214887817152443   |0.19704340532606415|236.0       |
|OGG|HNL|47662      |3.1249003398934163|3.0800637824682138  |0.16071503503839538|100.0       |
|LAS|LAX|47614      |10.985949510648128|5.746146091485698   |0.20139034737682193|236.0       |
|LGA|ORD|46511      |13.666057491776138|4.322504353808776   |0.20261873535292727|733.0       |
|ORD|LGA|46416      |

## 5. Build Nodes — Airports with Traffic & Delay Metrics

For each airport we compute basic graph metrics directly from the edges:
- `out_degree` — number of unique destinations reachable
- `in_degree` — number of unique origins that fly in
- `total_out_flights`, `total_in_flights`
- `avg_out_dep_delay`, `avg_in_arr_delay`

Then we enrich with OpenFlights metadata.

In [ ]:
# Outbound aggregates (from flights where this airport is the origin)
out_stats = flights.groupBy("ORIGIN").agg(
    F.countDistinct("DEST").alias("out_degree"),
    F.count("*").alias("total_out_flights"),
    F.avg("DEP_DELAY").alias("avg_out_dep_delay"),
    F.avg("ARR_DEL15").alias("out_pct_delayed_15")
).withColumnRenamed("ORIGIN", "iata")

# Inbound aggregates (from flights where this airport is the destination)
in_stats = flights.groupBy("DEST").agg(
    F.countDistinct("ORIGIN").alias("in_degree"),
    F.count("*").alias("total_in_flights"),
    F.avg("ARR_DELAY").alias("avg_in_arr_delay"),
    F.avg("ARR_DEL15").alias("in_pct_delayed_15")
).withColumnRenamed("DEST", "iata")

# Join outbound + inbound
nodes = out_stats.join(in_stats, on="iata", how="outer")

print(f"Unique airports (nodes): {nodes.count():,}")
nodes.show(5, truncate=False)

Unique airports (nodes): 381
+----+----------+-----------------+------------------+-------------------+---------+----------------+--------------------+-------------------+
|iata|out_degree|total_out_flights|avg_out_dep_delay |out_pct_delayed_15 |in_degree|total_in_flights|avg_in_arr_delay    |in_pct_delayed_15  |
+----+----------+-----------------+------------------+-------------------+---------+----------------+--------------------+-------------------+
|ABE |15        |21205            |9.248526290969112 |0.1571799103984909 |17       |21223           |5.314847099844508   |0.1823022192903925 |
|ABI |2         |9410             |7.094580233793836 |0.16992561105207227|2        |9417            |2.861633216523309   |0.14664967611765956|
|ABQ |30        |97509            |8.729317293788265 |0.15964680183367688|30       |97773           |4.674071573951909   |0.19284465036359733|
|ABR |1         |3562             |10.709713644020214|0.11341942728804043|1        |3551            |-3.001408054

In [ ]:
# Enrich nodes with OpenFlights metadata
nodes_enriched = nodes.join(airports_us, on="iata", how="left")

# Check how many airports matched
matched = nodes_enriched.filter(F.col("name").isNotNull()).count()
unmatched = nodes_enriched.filter(F.col("name").isNull()).count()
print(f"Airports with OpenFlights metadata: {matched}")
print(f"Airports without metadata:          {unmatched}")

if unmatched > 0:
    print("\nUnmatched airports (likely territories or small regional airports):")
    nodes_enriched.filter(F.col("name").isNull()).select("iata", "total_out_flights").show(20)

Airports with OpenFlights metadata: 370
Airports without metadata:          11

Unmatched airports (likely territories or small regional airports):
+----+-----------------+
|iata|total_out_flights|
+----+-----------------+
| BQN|             9480|
| EAR|             2840|
| GUM|             3554|
| IFP|             NULL|
| PPG|              345|
| PSE|             3116|
| SJU|           135894|
| SPN|             1738|
| STT|            23169|
| STX|             5320|
| XWA|             4965|
+----+-----------------+



## 6. Sanity Checks — Top Airports & Routes

In [ ]:
# Top 10 busiest airports by total outbound flights
print("Top 10 busiest airports (outbound flights):")
nodes_enriched.orderBy(F.desc("total_out_flights")).select(
    "iata", "name", "city", "total_out_flights", "out_degree", "avg_out_dep_delay"
).show(10, truncate=False)

Top 10 busiest airports (outbound flights):
+----+------------------------------------------------+-----------------+-----------------+----------+------------------+
|iata|name                                            |city             |total_out_flights|out_degree|avg_out_dep_delay |
+----+------------------------------------------------+-----------------+-----------------+----------+------------------+
|ATL |Hartsfield Jackson Atlanta International Airport|Atlanta          |1590429          |175       |8.192977492236372 |
|DFW |Dallas Fort Worth International Airport         |Dallas-Fort Worth|1327965          |201       |12.51740821482494 |
|ORD |Chicago O'Hare International Airport            |Chicago          |1249272          |199       |11.117877451827944|
|DEN |Denver International Airport                    |Denver           |1236560          |205       |13.10540531797891 |
|CLT |Charlotte Douglas International Airport         |Charlotte        |969852           |155       |

In [ ]:
# Top 10 most delayed airports (among airports with >10k flights to avoid tiny regional noise)
print("Top 10 most delayed airports (min 10k outbound flights):")
nodes_enriched \
    .filter(F.col("total_out_flights") >= 10000) \
    .orderBy(F.desc("avg_out_dep_delay")) \
    .select("iata", "name", "city", "total_out_flights", "avg_out_dep_delay", "out_pct_delayed_15") \
    .show(10, truncate=False)

Top 10 most delayed airports (min 10k outbound flights):
+----+-----------------------------------------------+---------------+-----------------+------------------+-------------------+
|iata|name                                           |city           |total_out_flights|avg_out_dep_delay |out_pct_delayed_15 |
+----+-----------------------------------------------+---------------+-----------------+------------------+-------------------+
|ASE |Aspen-Pitkin Co/Sardy Field                    |Aspen          |27660            |22.79005784526392 |0.2849602313810557 |
|EGE |Eagle County Regional Airport                  |Vail           |12583            |17.869665421600573|0.22903917984582373|
|HPN |Westchester County Airport                     |White Plains   |45728            |15.430195941217635|0.21693491952414276|
|EWR |Newark Liberty International Airport           |Newark         |541910           |15.095453119521691|0.2343008986732114 |
|JAC |Jackson Hole Airport                     

In [ ]:
# Top 10 busiest routes
print("Top 10 busiest routes:")
edges.orderBy(F.desc("num_flights")).show(10, truncate=False)

Top 10 busiest routes:
+---+---+-----------+------------------+--------------------+-------------------+------------+
|src|dst|num_flights|avg_dep_delay     |avg_arr_delay       |pct_delayed_15     |avg_distance|
+---+---+-----------+------------------+--------------------+-------------------+------------+
|SFO|LAX|54453      |8.016619837290875 |-0.05336712394174793|0.16316823682809029|337.0       |
|LAX|SFO|54442      |8.854487344329746 |2.3762352595422653  |0.18985342199037508|337.0       |
|HNL|OGG|47734      |3.3383961117861483|2.6832446474211253  |0.13265177860644403|100.0       |
|LAX|LAS|47690      |9.671146990983434 |5.214887817152443   |0.19704340532606415|236.0       |
|OGG|HNL|47662      |3.1249003398934163|3.0800637824682138  |0.16071503503839538|100.0       |
|LAS|LAX|47614      |10.985949510648128|5.746146091485698   |0.20139034737682193|236.0       |
|LGA|ORD|46511      |13.666057491776138|4.322504353808776   |0.20261873535292727|733.0       |
|ORD|LGA|46416      |14.551

In [ ]:
# Top 10 most delayed routes (among routes with enough traffic)
print("Top 10 most delayed routes (min 1000 flights):")
edges \
    .filter(F.col("num_flights") >= 1000) \
    .orderBy(F.desc("pct_delayed_15")) \
    .show(10, truncate=False)

Top 10 most delayed routes (min 1000 flights):
+---+---+-----------+------------------+------------------+-------------------+-----------------+
|src|dst|num_flights|avg_dep_delay     |avg_arr_delay     |pct_delayed_15     |avg_distance     |
+---+---+-----------+------------------+------------------+-------------------+-----------------+
|EWR|BQN|1409       |33.83392476933996 |27.990773598296663|0.43222143364088006|1585.0           |
|BDL|SJU|3177       |27.86024551463645 |27.845451683978595|0.43090966320428076|1666.0           |
|SJU|DCA|1567       |32.13146139119336 |30.639438417358008|0.4211869814932993 |1554.0           |
|BWI|LGA|1055       |21.99526066350711 |18.635071090047393|0.4                |185.0            |
|USA|FLL|1526       |29.146788990825687|30.844692005242464|0.3984272608125819 |643.0            |
|SJU|BDL|3176       |27.66435768261965 |26.507556675062972|0.39074307304785894|1666.0           |
|EWR|SMF|1132       |26.909010600706715|19.364840989399294|0.3904593639

## 7. Save Nodes & Edges as Parquet

In [ ]:
NODES_PATH = "/content/drive/MyDrive/Big Data Project/parquet/graph_nodes"
EDGES_PATH = "/content/drive/MyDrive/Big Data Project/parquet/graph_edges"

nodes_enriched.write.mode("overwrite").parquet(NODES_PATH)
edges.write.mode("overwrite").parquet(EDGES_PATH)

print(f"Saved nodes to: {NODES_PATH}")
print(f"Saved edges to: {EDGES_PATH}")

Saved nodes to: /content/drive/MyDrive/Big Data Project/parquet/graph_nodes
Saved edges to: /content/drive/MyDrive/Big Data Project/parquet/graph_edges


In [ ]:
# Verify
nodes_verify = spark.read.parquet(NODES_PATH)
edges_verify = spark.read.parquet(EDGES_PATH)
print(f"Nodes: {nodes_verify.count():,}")
print(f"Edges: {edges_verify.count():,}")

Nodes: 381
Edges: 8,128


## Summary

**What we did:**
- Loaded the cleaned flight Parquet from Step 1
- Loaded and filtered OpenFlights metadata to US airports
- Built **edges** — one row per directed route with flight count and delay metrics
- Built **nodes** — one row per airport with in/out degree, traffic volume, and delay stats
- Joined node stats with OpenFlights metadata (name, city, lat/lon)
- Saved both as Parquet for downstream use

**Next step:** Graph analytics — centrality measures (PageRank, betweenness-style metrics) using GraphFrames or custom Spark logic to find delay-critical airports and fragile routes.

In [ ]:
spark.stop()
print("Spark session stopped. Step 2 complete!")

Spark session stopped. Step 2 complete!
